# Сравнительный анализ подходов к распознаванию показаний счётчиков

Данный ноутбук проводит **реальные эксперименты** по распознаванию цифр на изображениях приборов учёта.

## Сравниваемые подходы:

| # | Подход | Описание |
|---|---|---|
| A | **Tesseract OCR** | Классический OCR с различными режимами PSM |
| B | **EasyOCR** | Нейросетевой OCR (CRAFT + CRNN) |
| C | **PaddleOCR** | OCR от Baidu (DB + SVTR) |
| D | **CRNN + CTC** | Собственная CRNN модель (ResNet18 + BiLSTM + CTC) |
| E | **YOLOv8n (from scratch)** | Обучение с нуля, без предобученных весов |
| F | **YOLOv8n (full fine-tune)** | Все слои обучаются, COCO pretrained |
| G | **YOLOv8n (transfer, freeze=10)** | Backbone заморожен, только head обучается |

## Метрики:
- **Exact Match (EM)** — показание целиком совпадает с ground truth
- **Character Accuracy** — посимвольная точность
- **mAP@50** — для detection-моделей
- **Inference time** — время обработки одного изображения

---
⚡ **Перед запуском**: Runtime → Change runtime type → **T4 GPU**

---
# Часть 0. Установка и настройка

In [ ]:
%%time
!pip install ultralytics roboflow pytesseract easyocr paddlepaddle paddleocr -q
!apt-get install -y tesseract-ocr > /dev/null 2>&1
print("Установка завершена.")

In [ ]:
import os
import glob
import time
import yaml
import random
import shutil
import warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageFilter, ImageEnhance
from collections import Counter, defaultdict
from IPython.display import display, Image as IPImage, HTML
import cv2
import torch

warnings.filterwarnings("ignore")

# Фиксируем seed для воспроизводимости
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

---
# Часть 1. Загрузка датасета

In [ ]:
from roboflow import Roboflow

# === ВСТАВЬТЕ ВАШ API-КЛЮЧ ROBOFLOW ===
ROBOFLOW_API_KEY = "YOUR_API_KEY"  # <-- замените

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("unilogic").project("ocr-meter-reading")
dataset = project.version(2).download("yolov8")

DATASET_DIR = dataset.location
print(f"Датасет: {DATASET_DIR}")

In [ ]:
# Загрузим конфигурацию и прочитаем все аннотации
data_yaml_path = os.path.join(DATASET_DIR, "data.yaml")
with open(data_yaml_path, "r") as f:
    data_config = yaml.safe_load(f)

CLASS_NAMES = data_config["names"]
print(f"Классы ({len(CLASS_NAMES)}): {CLASS_NAMES}")

# Обновляем пути для Colab
data_config["path"] = DATASET_DIR
data_config["train"] = "train/images"
data_config["val"] = "valid/images"
data_config["test"] = "test/images"
with open(data_yaml_path, "w") as f:
    yaml.dump(data_config, f, default_flow_style=False)

# Считаем изображения
for split in ["train", "valid", "test"]:
    imgs = glob.glob(os.path.join(DATASET_DIR, split, "images", "*"))
    print(f"{split}: {len(imgs)} изображений")

In [ ]:
def get_ground_truth(label_path, class_names):
    """
    Читает YOLO-аннотацию и возвращает ground truth показание.
    Цифры сортируются по x-координате (слева направо).
    """
    if not os.path.exists(label_path):
        return ""
    detections = []
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                cls_id = int(parts[0])
                cx = float(parts[1])
                detections.append((cx, cls_id))
    detections.sort(key=lambda d: d[0])
    name_list = class_names if isinstance(class_names, list) else list(range(10))
    return "".join(str(name_list[d[1]]) for d in detections)


def build_test_set(dataset_dir, class_names, split="test"):
    """
    Формирует список (image_path, ground_truth_reading) для оценки.
    """
    img_dir = os.path.join(dataset_dir, split, "images")
    lbl_dir = os.path.join(dataset_dir, split, "labels")
    test_data = []
    for img_path in sorted(glob.glob(os.path.join(img_dir, "*"))):
        basename = os.path.splitext(os.path.basename(img_path))[0]
        label_path = os.path.join(lbl_dir, basename + ".txt")
        gt = get_ground_truth(label_path, class_names)
        if gt:  # только если есть аннотация
            test_data.append((img_path, gt))
    return test_data


test_set = build_test_set(DATASET_DIR, CLASS_NAMES)
print(f"\nТестовый набор: {len(test_set)} изображений с GT-показаниями")
print(f"Примеры GT: {[t[1] for t in test_set[:10]]}")

In [ ]:
def compute_metrics(predictions, ground_truths):
    """
    Вычисляет метрики для OCR-подходов.
    predictions: list[str] — предсказанные строки
    ground_truths: list[str] — эталонные строки
    """
    assert len(predictions) == len(ground_truths)
    n = len(predictions)

    exact_match = 0
    total_chars = 0
    correct_chars = 0
    off_by_one = 0  # ошибка в 1 символе

    for pred, gt in zip(predictions, ground_truths):
        pred_clean = "".join(c for c in str(pred) if c.isdigit())

        if pred_clean == gt:
            exact_match += 1

        # Посимвольная точность
        max_len = max(len(pred_clean), len(gt))
        total_chars += max_len
        for i in range(min(len(pred_clean), len(gt))):
            if pred_clean[i] == gt[i]:
                correct_chars += 1

        # Off-by-one: отличается ровно на 1 символ
        if len(pred_clean) == len(gt):
            diffs = sum(1 for a, b in zip(pred_clean, gt) if a != b)
            if diffs == 1:
                off_by_one += 1

    return {
        "exact_match": exact_match / n * 100,
        "char_accuracy": correct_chars / max(total_chars, 1) * 100,
        "off_by_one": off_by_one / n * 100,
        "total": n,
        "correct": exact_match
    }


def print_metrics(name, metrics, inference_time=None):
    """Красивый вывод метрик."""
    print(f"\n{'='*55}")
    print(f"  {name}")
    print(f"{'='*55}")
    print(f"  Exact Match:      {metrics['exact_match']:.1f}%  ({metrics['correct']}/{metrics['total']})")
    print(f"  Char Accuracy:    {metrics['char_accuracy']:.1f}%")
    print(f"  Off-by-one:       {metrics['off_by_one']:.1f}%")
    if inference_time:
        print(f"  Avg Inference:    {inference_time:.0f} мс/изображение")
    print(f"{'='*55}")

---
# Часть 2. Визуализация тестовых изображений

Посмотрим на данные, с которыми будем работать.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
samples = random.sample(test_set, 8)

for ax, (img_path, gt) in zip(axes.flat, samples):
    img = Image.open(img_path)
    ax.imshow(img)
    ax.set_title(f"GT: {gt}", fontsize=14, fontweight="bold", color="green")
    ax.axis("off")

plt.suptitle("Примеры тестовых изображений с ground truth", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

---
# Часть 3. Эксперимент A — Tesseract OCR

Tesseract — классический OCR-движок (Google, открытый исходный код).
Тестируем различные режимы Page Segmentation Mode (PSM) и предобработки.

In [ ]:
import pytesseract

def preprocess_for_ocr(img, method="none"):
    """
    Различные методы предобработки изображения для OCR.
    """
    if method == "none":
        return img

    elif method == "grayscale":
        return img.convert("L")

    elif method == "threshold":
        gray = np.array(img.convert("L"))
        _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        return Image.fromarray(binary)

    elif method == "adaptive":
        gray = np.array(img.convert("L"))
        binary = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                       cv2.THRESH_BINARY, 11, 2)
        return Image.fromarray(binary)

    elif method == "sharpen_contrast":
        img = ImageEnhance.Contrast(img).enhance(2.0)
        img = ImageEnhance.Sharpness(img).enhance(2.0)
        return img.convert("L")

    elif method == "denoise":
        gray = np.array(img.convert("L"))
        denoised = cv2.fastNlMeansDenoising(gray, None, 10, 7, 21)
        _, binary = cv2.threshold(denoised, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        return Image.fromarray(binary)

    return img


def tesseract_recognize(img_path, psm=7, preprocess="none"):
    """
    Распознавание через Tesseract.
    PSM modes:
      6 — Assume a single uniform block of text
      7 — Treat the image as a single text line
      8 — Treat the image as a single word
      13 — Raw line
    """
    img = Image.open(img_path)
    img = preprocess_for_ocr(img, preprocess)
    config = f"--psm {psm} --oem 3 -c tessedit_char_whitelist=0123456789"
    text = pytesseract.image_to_string(img, config=config)
    return "".join(c for c in text if c.isdigit())


print("Tesseract version:", pytesseract.get_tesseract_version())

In [ ]:
%%time
# Тестируем все комбинации PSM + предобработка

psm_modes = [6, 7, 8, 13]
preprocess_methods = ["none", "grayscale", "threshold", "adaptive", "sharpen_contrast", "denoise"]

tesseract_results = {}

print("Тестирование Tesseract OCR...")
print(f"Комбинаций: {len(psm_modes)} PSM × {len(preprocess_methods)} preprocess = {len(psm_modes)*len(preprocess_methods)}")
print()

for psm in psm_modes:
    for preprocess in preprocess_methods:
        key = f"PSM={psm}, {preprocess}"
        predictions = []
        ground_truths = []
        t_start = time.time()

        for img_path, gt in test_set:
            try:
                pred = tesseract_recognize(img_path, psm=psm, preprocess=preprocess)
            except Exception:
                pred = ""
            predictions.append(pred)
            ground_truths.append(gt)

        elapsed = (time.time() - t_start) * 1000 / len(test_set)
        metrics = compute_metrics(predictions, ground_truths)
        tesseract_results[key] = {**metrics, "inference_ms": elapsed}

        print(f"  {key:<35} EM={metrics['exact_match']:5.1f}%  Char={metrics['char_accuracy']:5.1f}%  ({elapsed:.0f} мс)")

# Лучший результат
best_key = max(tesseract_results, key=lambda k: tesseract_results[k]["exact_match"])
best = tesseract_results[best_key]
print(f"\n★ Лучшая комбинация: {best_key}")
print_metrics(f"Tesseract — {best_key}", best, best["inference_ms"])

In [ ]:
# Визуализация результатов Tesseract по комбинациям
fig, ax = plt.subplots(figsize=(14, 6))

sorted_results = sorted(tesseract_results.items(), key=lambda x: x[1]["exact_match"], reverse=True)
keys = [k for k, _ in sorted_results[:12]]  # top-12
em_vals = [tesseract_results[k]["exact_match"] for k in keys]
char_vals = [tesseract_results[k]["char_accuracy"] for k in keys]

x = np.arange(len(keys))
w = 0.35
bars1 = ax.bar(x - w/2, em_vals, w, label="Exact Match", color="#e74c3c", edgecolor="black")
bars2 = ax.bar(x + w/2, char_vals, w, label="Char Accuracy", color="#f39c12", edgecolor="black")

ax.set_ylabel("Точность, %", fontsize=12)
ax.set_title("Tesseract OCR: сравнение комбинаций PSM + предобработка", fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(keys, rotation=45, ha="right", fontsize=8)
ax.legend(fontsize=11)
ax.set_ylim(0, 100)
ax.axhline(y=50, color="gray", linestyle="--", alpha=0.5)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f"{bar.get_height():.1f}", ha="center", fontsize=7)

plt.tight_layout()
plt.show()

---
# Часть 4. Эксперимент B — EasyOCR

EasyOCR — нейросетевой OCR (архитектура: CRAFT text detector + CRNN recognizer).
Поддерживает 80+ языков, GPU-ускорение.

In [ ]:
import easyocr

reader = easyocr.Reader(["en"], gpu=torch.cuda.is_available())
print("EasyOCR загружен.")

In [ ]:
%%time
# Тестирование EasyOCR с разными настройками

easyocr_configs = [
    {"name": "default",         "kwargs": {}},
    {"name": "low_text=0.3",    "kwargs": {"low_text": 0.3}},
    {"name": "low_text=0.2",    "kwargs": {"low_text": 0.2}},
    {"name": "allowlist digits", "kwargs": {"allowlist": "0123456789"}},
    {"name": "contrast+digits",  "kwargs": {"allowlist": "0123456789", "contrast_ths": 0.3}},
]

easyocr_results = {}

print("Тестирование EasyOCR...")
for config in easyocr_configs:
    predictions = []
    ground_truths = []
    t_start = time.time()

    for img_path, gt in test_set:
        try:
            result = reader.readtext(img_path, **config["kwargs"])
            # Собираем все распознанные тексты, оставляем только цифры
            all_text = " ".join([r[1] for r in result])
            digits = "".join(c for c in all_text if c.isdigit())
            # Берём самую длинную последовательность цифр из результатов
            digit_sequences = []
            for r in result:
                seq = "".join(c for c in r[1] if c.isdigit())
                if seq:
                    digit_sequences.append(seq)
            if digit_sequences:
                pred = max(digit_sequences, key=len)
            else:
                pred = digits[:10] if digits else ""
        except Exception:
            pred = ""
        predictions.append(pred)
        ground_truths.append(gt)

    elapsed = (time.time() - t_start) * 1000 / len(test_set)
    metrics = compute_metrics(predictions, ground_truths)
    easyocr_results[config["name"]] = {**metrics, "inference_ms": elapsed}

    print(f"  {config['name']:<25} EM={metrics['exact_match']:5.1f}%  Char={metrics['char_accuracy']:5.1f}%  ({elapsed:.0f} мс)")

best_key = max(easyocr_results, key=lambda k: easyocr_results[k]["exact_match"])
best = easyocr_results[best_key]
print_metrics(f"EasyOCR — {best_key}", best, best["inference_ms"])

---
# Часть 5. Эксперимент C — PaddleOCR

PaddleOCR (Baidu) — ещё один нейросетевой OCR. Архитектура: DB text detector + SVTR recognizer.
Часто показывает лучшие результаты, чем EasyOCR на структурированном тексте.

In [ ]:
from paddleocr import PaddleOCR

# Инициализируем PaddleOCR
paddle_ocr = PaddleOCR(use_angle_cls=True, lang="en", show_log=False)
print("PaddleOCR загружен.")

In [ ]:
%%time
# Тестирование PaddleOCR

paddle_predictions = []
paddle_gts = []
t_start = time.time()

print("Тестирование PaddleOCR...")
for img_path, gt in test_set:
    try:
        result = paddle_ocr.ocr(img_path, cls=True)
        # PaddleOCR возвращает [[line, (text, conf)], ...]
        digit_sequences = []
        if result and result[0]:
            for line in result[0]:
                text = line[1][0]
                seq = "".join(c for c in text if c.isdigit())
                if seq:
                    digit_sequences.append(seq)
        if digit_sequences:
            pred = max(digit_sequences, key=len)
        else:
            pred = ""
    except Exception:
        pred = ""
    paddle_predictions.append(pred)
    paddle_gts.append(gt)

paddle_elapsed = (time.time() - t_start) * 1000 / len(test_set)
paddle_metrics = compute_metrics(paddle_predictions, paddle_gts)
print_metrics("PaddleOCR (default)", paddle_metrics, paddle_elapsed)

---
# Часть 6. Визуализация ошибок OCR

Посмотрим, **почему** классические OCR ошибаются — типичные проблемы.

In [ ]:
# Находим примеры ошибок для лучшей конфигурации Tesseract
best_tess_key = max(tesseract_results, key=lambda k: tesseract_results[k]["exact_match"])
psm_val = int(best_tess_key.split(",")[0].split("=")[1])
preprocess_val = best_tess_key.split(", ")[1]

# Собираем ошибки всех трёх OCR
error_examples = []  # (img_path, gt, tesseract_pred, easyocr_pred, paddle_pred)

best_easy_key = max(easyocr_results, key=lambda k: easyocr_results[k]["exact_match"])

for i, (img_path, gt) in enumerate(test_set):
    tess_pred = tesseract_recognize(img_path, psm=psm_val, preprocess=preprocess_val)

    # EasyOCR
    try:
        cfg = next(c for c in easyocr_configs if c["name"] == best_easy_key)
        result = reader.readtext(img_path, **cfg["kwargs"])
        seqs = ["".join(c for c in r[1] if c.isdigit()) for r in result]
        seqs = [s for s in seqs if s]
        easy_pred = max(seqs, key=len) if seqs else ""
    except Exception:
        easy_pred = ""

    # PaddleOCR
    paddle_pred = paddle_predictions[i]

    # Если все три ошиблись — интересный пример
    tess_clean = "".join(c for c in tess_pred if c.isdigit())
    easy_clean = "".join(c for c in easy_pred if c.isdigit())
    paddle_clean = "".join(c for c in paddle_pred if c.isdigit())

    if tess_clean != gt or easy_clean != gt or paddle_clean != gt:
        error_examples.append((img_path, gt, tess_clean, easy_clean, paddle_clean))

    if len(error_examples) >= 30:
        break

print(f"Найдено примеров ошибок: {len(error_examples)}")

In [ ]:
# Визуализация ошибок OCR
n_show = min(8, len(error_examples))
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for ax, (img_path, gt, tess, easy, paddle) in zip(axes.flat, error_examples[:n_show]):
    img = Image.open(img_path)
    ax.imshow(img)

    title_lines = [
        f"GT: {gt}",
        f"Tess: {tess or '∅'}" + (" ✓" if tess == gt else " ✗"),
        f"Easy: {easy or '∅'}" + (" ✓" if easy == gt else " ✗"),
        f"Paddle: {paddle or '∅'}" + (" ✓" if paddle == gt else " ✗"),
    ]
    title = "\n".join(title_lines)
    ax.set_title(title, fontsize=9, family="monospace", loc="left")
    ax.axis("off")

for ax in axes.flat[n_show:]:
    ax.axis("off")

plt.suptitle("Типичные ошибки OCR-движков на изображениях счётчиков", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Анализ типов ошибок OCR
# Для лучшего Tesseract
confusion_pairs = Counter()  # (predicted_digit, actual_digit)
extra_digits = 0  # лишние цифры
missing_digits = 0  # пропущенные цифры
wrong_length = 0

for img_path, gt in test_set:
    pred = tesseract_recognize(img_path, psm=psm_val, preprocess=preprocess_val)
    pred_clean = "".join(c for c in pred if c.isdigit())

    if len(pred_clean) != len(gt):
        wrong_length += 1
        if len(pred_clean) > len(gt):
            extra_digits += 1
        else:
            missing_digits += 1
    else:
        for p, g in zip(pred_clean, gt):
            if p != g:
                confusion_pairs[(g, p)] += 1

print("=== Анализ ошибок Tesseract ===")
print(f"Неверная длина показания: {wrong_length}/{len(test_set)} ({wrong_length/len(test_set)*100:.1f}%)")
print(f"  — Лишние цифры (серийник, kWh): {extra_digits}")
print(f"  — Пропущенные цифры: {missing_digits}")
print(f"\nТоп-10 путаниц (GT → Predicted):")
for (gt_c, pred_c), count in confusion_pairs.most_common(10):
    print(f"  '{gt_c}' → '{pred_c}': {count} раз")

---
# Часть 7. Эксперимент D — CRNN + CTC Loss

Обучим собственную CRNN-модель (Convolutional Recurrent Neural Network):
- **Encoder**: ResNet-18 (pretrained ImageNet)
- **Sequence modeling**: BiLSTM
- **Decoder**: CTC (Connectionist Temporal Classification)

Это популярный подход для распознавания текстовых строк (license plates, captcha).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models


class MeterCRNNDataset(Dataset):
    """
    Датасет для CRNN: загружает изображение и GT-показание.
    Изображение ресайзится до фиксированного размера (32×160).
    """
    CHARS = "0123456789"
    BLANK = 10  # CTC blank label

    def __init__(self, data_list, transform=None):
        self.data = data_list
        self.transform = transform or transforms.Compose([
            transforms.Resize((32, 160)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, gt_text = self.data[idx]
        img = Image.open(img_path).convert("RGB")
        img = self.transform(img)

        # Encode text → indices
        target = [self.CHARS.index(c) for c in gt_text if c in self.CHARS]
        target_len = len(target)

        return img, torch.tensor(target, dtype=torch.long), target_len


def crnn_collate(batch):
    """Custom collate для переменной длины targets."""
    images, targets, target_lens = zip(*batch)
    images = torch.stack(images)
    targets = torch.cat(targets)
    target_lens = torch.tensor(target_lens, dtype=torch.long)
    return images, targets, target_lens


class CRNN(nn.Module):
    """
    CRNN: CNN (ResNet-18 backbone) + BiLSTM + Linear.
    """
    def __init__(self, num_classes=11, hidden_size=256):
        super().__init__()
        # Feature extractor (ResNet-18, first 4 layers)
        resnet = models.resnet18(weights="IMAGENET1K_V1")
        self.cnn = nn.Sequential(
            resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool,
            resnet.layer1,
            resnet.layer2,
            resnet.layer3,
            # Adaptive pool to fixed height=1
            nn.AdaptiveAvgPool2d((1, None))
        )
        # RNN
        self.rnn = nn.LSTM(
            input_size=256,  # ResNet layer3 output channels
            hidden_size=hidden_size,
            num_layers=2,
            bidirectional=True,
            batch_first=True,
            dropout=0.3
        )
        # Classifier
        self.fc = nn.Linear(hidden_size * 2, num_classes)  # *2 for bidirectional

    def forward(self, x):
        # x: (B, 3, 32, 160)
        features = self.cnn(x)          # (B, 256, 1, W')
        features = features.squeeze(2)   # (B, 256, W')
        features = features.permute(0, 2, 1)  # (B, W', 256)
        rnn_out, _ = self.rnn(features)  # (B, W', 512)
        output = self.fc(rnn_out)         # (B, W', 11)
        return output.permute(1, 0, 2)    # (W', B, 11) for CTC


print("CRNN модель определена.")

In [ ]:
# Готовим данные для CRNN
train_data = build_test_set(DATASET_DIR, CLASS_NAMES, split="train")
val_data = build_test_set(DATASET_DIR, CLASS_NAMES, split="valid")

print(f"CRNN train: {len(train_data)}, val: {len(val_data)}, test: {len(test_set)}")

train_dataset = MeterCRNNDataset(train_data)
val_dataset = MeterCRNNDataset(val_data)
test_dataset = MeterCRNNDataset(test_set)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,
                           collate_fn=crnn_collate, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False,
                         collate_fn=crnn_collate, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False,
                          collate_fn=crnn_collate, num_workers=2)

In [ ]:
# Обучение CRNN
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
crnn_model = CRNN(num_classes=11).to(device)  # 10 digits + blank
criterion = nn.CTCLoss(blank=10, zero_infinity=True)
optimizer = optim.AdamW(crnn_model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

CRNN_EPOCHS = 30
history = {"train_loss": [], "val_loss": []}

print(f"Обучение CRNN на {device}...")
print(f"Параметров: {sum(p.numel() for p in crnn_model.parameters()):,}")

best_val_loss = float("inf")

for epoch in range(CRNN_EPOCHS):
    # Train
    crnn_model.train()
    train_loss = 0.0
    for images, targets, target_lens in train_loader:
        images = images.to(device)
        targets = targets.to(device)

        outputs = crnn_model(images)  # (T, B, C)
        input_lens = torch.full((images.size(0),), outputs.size(0), dtype=torch.long)

        loss = criterion(outputs.log_softmax(2), targets, input_lens, target_lens)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(crnn_model.parameters(), max_norm=5.0)
        optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_loader)
    scheduler.step()

    # Validation
    crnn_model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, targets, target_lens in val_loader:
            images = images.to(device)
            targets = targets.to(device)
            outputs = crnn_model(images)
            input_lens = torch.full((images.size(0),), outputs.size(0), dtype=torch.long)
            loss = criterion(outputs.log_softmax(2), targets, input_lens, target_lens)
            val_loss += loss.item()

    val_loss /= len(val_loader)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(crnn_model.state_dict(), "crnn_best.pth")

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"  Epoch {epoch+1:2d}/{CRNN_EPOCHS}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}")

print(f"\nЛучший val_loss: {best_val_loss:.4f}")

In [ ]:
# Learning curves для CRNN
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(history["train_loss"], label="Train Loss", linewidth=2)
ax.plot(history["val_loss"], label="Val Loss", linewidth=2)
ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("CTC Loss", fontsize=12)
ax.set_title("CRNN (ResNet18 + BiLSTM + CTC): кривые обучения", fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Оценка CRNN на тесте
def ctc_decode(output):
    """Greedy CTC decode: argmax + remove blanks + remove duplicates."""
    CHARS = "0123456789"
    BLANK = 10
    indices = output.argmax(dim=-1)  # (T,)
    decoded = []
    prev = BLANK
    for idx in indices:
        idx = idx.item()
        if idx != BLANK and idx != prev:
            if idx < len(CHARS):
                decoded.append(CHARS[idx])
        prev = idx
    return "".join(decoded)


# Загружаем лучшую модель
crnn_model.load_state_dict(torch.load("crnn_best.pth", weights_only=True))
crnn_model.eval()

crnn_predictions = []
crnn_gts = []
crnn_times = []

transform = transforms.Compose([
    transforms.Resize((32, 160)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

with torch.no_grad():
    for img_path, gt in test_set:
        img = Image.open(img_path).convert("RGB")
        tensor = transform(img).unsqueeze(0).to(device)

        t0 = time.time()
        output = crnn_model(tensor)  # (T, 1, 11)
        crnn_times.append((time.time() - t0) * 1000)

        pred = ctc_decode(output[:, 0, :])
        crnn_predictions.append(pred)
        crnn_gts.append(gt)

crnn_metrics = compute_metrics(crnn_predictions, crnn_gts)
crnn_avg_time = np.mean(crnn_times)
print_metrics("CRNN (ResNet18 + BiLSTM + CTC)", crnn_metrics, crnn_avg_time)

In [ ]:
# Визуализация ошибок CRNN
crnn_errors = [(img, gt, pred) for (img, gt), pred
               in zip(test_set, crnn_predictions)
               if "".join(c for c in pred if c.isdigit()) != gt]

print(f"Ошибок CRNN: {len(crnn_errors)}/{len(test_set)}")

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, (img_path, gt, pred) in zip(axes.flat, crnn_errors[:8]):
    img = Image.open(img_path)
    ax.imshow(img)
    color = "red"
    ax.set_title(f"GT: {gt}\nCRNN: {pred or '∅'}", fontsize=12, color=color)
    ax.axis("off")
for ax in axes.flat[len(crnn_errors[:8]):]:
    ax.axis("off")

plt.suptitle("Ошибки CRNN — модель путает цифры без точной локализации", fontsize=14)
plt.tight_layout()
plt.show()

---
# Часть 8. Эксперимент E/F/G — YOLOv8n (3 варианта)

Сравним три стратегии обучения YOLOv8n:
- **E) From Scratch** — без предобученных весов (`yolov8n.yaml`)
- **F) Full Fine-tuning** — все слои обучаются (`freeze=0`)
- **G) Transfer Learning** — backbone заморожен (`freeze=10`)

In [ ]:
from ultralytics import YOLO

YOLO_EPOCHS = 50
YOLO_PATIENCE = 15
YOLO_IMGSZ = 640
YOLO_BATCH = 16

yolo_experiments = [
    {
        "name": "YOLOv8n_scratch",
        "model_init": "yolov8n.yaml",   # только архитектура, без весов
        "freeze": 0,
        "lr0": 0.01,
        "epochs": YOLO_EPOCHS,
    },
    {
        "name": "YOLOv8n_full_finetune",
        "model_init": "yolov8n.pt",      # COCO pretrained
        "freeze": 0,
        "lr0": 0.0005,
        "epochs": YOLO_EPOCHS,
    },
    {
        "name": "YOLOv8n_transfer_freeze10",
        "model_init": "yolov8n.pt",      # COCO pretrained
        "freeze": 10,                     # backbone frozen
        "lr0": 0.001,
        "epochs": YOLO_EPOCHS,
    },
]

print(f"Будет запущено {len(yolo_experiments)} экспериментов YOLO, по {YOLO_EPOCHS} эпох каждый.")
print("Ориентировочное время: ~60–90 минут на T4 GPU.")

In [ ]:
%%time
# Обучение YOLOv8n — 3 эксперимента

yolo_results = {}

for exp in yolo_experiments:
    print(f"\n{'='*60}")
    print(f"  Эксперимент: {exp['name']}")
    print(f"  model={exp['model_init']}, freeze={exp['freeze']}, lr={exp['lr0']}")
    print(f"{'='*60}")

    model = YOLO(exp["model_init"])

    t_start = time.time()
    results = model.train(
        data=data_yaml_path,
        epochs=exp["epochs"],
        imgsz=YOLO_IMGSZ,
        batch=YOLO_BATCH,
        freeze=exp["freeze"],
        patience=YOLO_PATIENCE,
        optimizer="AdamW",
        lr0=exp["lr0"],
        cos_lr=True,
        device=0,
        project="experiments",
        name=exp["name"],
        verbose=False,
        plots=True,
        seed=SEED
    )
    train_time = time.time() - t_start

    yolo_results[exp["name"]] = {
        "train_time_min": train_time / 60,
        "results_dir": f"experiments/{exp['name']}",
        "best_path": f"experiments/{exp['name']}/weights/best.pt"
    }

    print(f"  Готово за {train_time/60:.1f} минут.")

In [ ]:
# Валидация каждой YOLO-модели на тестовом наборе

for exp_name, info in yolo_results.items():
    print(f"\n--- Валидация: {exp_name} ---")
    model = YOLO(info["best_path"])

    metrics = model.val(
        data=data_yaml_path,
        split="test",
        imgsz=YOLO_IMGSZ,
        batch=YOLO_BATCH,
        device=0,
        verbose=False
    )

    info["map50"] = metrics.box.map50
    info["map50_95"] = metrics.box.map
    info["precision"] = metrics.box.mp
    info["recall"] = metrics.box.mr

    print(f"  mAP@50={info['map50']:.4f}  mAP@50-95={info['map50_95']:.4f}  P={info['precision']:.4f}  R={info['recall']:.4f}")

In [ ]:
# Exact Match для YOLO-моделей (как OCR-метрика)

def yolo_predict_reading(model, img_path, conf=0.25, imgsz=640):
    """Предсказание показания: detect → sort by x → concat."""
    results = model.predict(img_path, conf=conf, imgsz=imgsz, verbose=False)
    result = results[0]
    if len(result.boxes) == 0:
        return ""
    detections = []
    for box in result.boxes:
        x1 = box.xyxy[0][0].item()
        cls_id = int(box.cls[0].item())
        detections.append((x1, cls_id))
    detections.sort(key=lambda d: d[0])
    return "".join(str(d[1]) for d in detections)


for exp_name, info in yolo_results.items():
    model = YOLO(info["best_path"])
    predictions = []
    gts = []
    times = []

    for img_path, gt in test_set:
        t0 = time.time()
        pred = yolo_predict_reading(model, img_path)
        times.append((time.time() - t0) * 1000)
        predictions.append(pred)
        gts.append(gt)

    em_metrics = compute_metrics(predictions, gts)
    info["exact_match"] = em_metrics["exact_match"]
    info["char_accuracy"] = em_metrics["char_accuracy"]
    info["inference_ms"] = np.mean(times)

    print_metrics(exp_name, em_metrics, info["inference_ms"])

In [ ]:
# Визуализация learning curves для 3 YOLO-экспериментов

import csv

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ["#e74c3c", "#2ecc71", "#3498db"]
plot_keys = [
    ("train/box_loss", "Train Box Loss"),
    ("metrics/mAP50(B)", "Val mAP@50"),
    ("metrics/precision(B)", "Val Precision"),
]

for ax, (csv_key, title) in zip(axes, plot_keys):
    for i, (exp_name, info) in enumerate(yolo_results.items()):
        csv_path = os.path.join(info["results_dir"], "results.csv")
        if os.path.exists(csv_path):
            epochs_data = []
            values = []
            with open(csv_path, "r") as f:
                reader_csv = csv.DictReader(f)
                for row in reader_csv:
                    # Strip whitespace from keys
                    row = {k.strip(): v.strip() for k, v in row.items()}
                    if csv_key.strip() in row:
                        epochs_data.append(int(row.get("epoch", len(epochs_data))))
                        values.append(float(row[csv_key.strip()]))

            short_name = exp_name.replace("YOLOv8n_", "")
            ax.plot(epochs_data, values, label=short_name,
                    color=colors[i], linewidth=2)

    ax.set_xlabel("Epoch", fontsize=11)
    ax.set_title(title, fontsize=13)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.suptitle("YOLOv8n: сравнение стратегий обучения", fontsize=15, y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
# Confusion Matrix для лучшей YOLO-модели
best_yolo = max(yolo_results, key=lambda k: yolo_results[k].get("map50", 0))
cm_path = os.path.join(yolo_results[best_yolo]["results_dir"], "confusion_matrix_normalized.png")
if os.path.exists(cm_path):
    print(f"Confusion matrix: {best_yolo}")
    display(IPImage(filename=cm_path, width=600))
else:
    cm_path = os.path.join(yolo_results[best_yolo]["results_dir"], "confusion_matrix.png")
    if os.path.exists(cm_path):
        display(IPImage(filename=cm_path, width=600))

---
# Часть 9. Визуальное сравнение предсказаний

Сравним, как разные подходы справляются с одними и теми же изображениями.

In [ ]:
# Получаем предсказания лучшей YOLO-модели
best_yolo_name = max(yolo_results, key=lambda k: yolo_results[k].get("exact_match", 0))
best_yolo_model = YOLO(yolo_results[best_yolo_name]["best_path"])

# Лучший Tesseract
best_tess_key = max(tesseract_results, key=lambda k: tesseract_results[k]["exact_match"])
tess_psm = int(best_tess_key.split(",")[0].split("=")[1])
tess_pp = best_tess_key.split(", ")[1]

# Лучший EasyOCR
best_easy_key = max(easyocr_results, key=lambda k: easyocr_results[k]["exact_match"])
best_easy_cfg = next(c for c in easyocr_configs if c["name"] == best_easy_key)


# Собираем предсказания для 12 примеров
comparison_samples = random.sample(test_set, 12)

fig, axes = plt.subplots(3, 4, figsize=(22, 16))

for ax, (img_path, gt) in zip(axes.flat, comparison_samples):
    img = Image.open(img_path)
    ax.imshow(img)

    # YOLO prediction
    yolo_pred = yolo_predict_reading(best_yolo_model, img_path)

    # Tesseract
    tess_pred = tesseract_recognize(img_path, psm=tess_psm, preprocess=tess_pp)

    # EasyOCR
    try:
        result = reader.readtext(img_path, **best_easy_cfg["kwargs"])
        seqs = ["".join(c for c in r[1] if c.isdigit()) for r in result]
        seqs = [s for s in seqs if s]
        easy_pred = max(seqs, key=len) if seqs else ""
    except Exception:
        easy_pred = ""

    # CRNN
    crnn_idx = next((i for i, (p, g) in enumerate(test_set) if p == img_path), None)
    crnn_pred = crnn_predictions[crnn_idx] if crnn_idx is not None else "?"

    def mark(pred, gt):
        clean = "".join(c for c in str(pred) if c.isdigit())
        return f"{pred or '∅'} {'✓' if clean == gt else '✗'}"

    lines = [
        f"GT:     {gt}",
        f"YOLO:   {mark(yolo_pred, gt)}",
        f"Tess:   {mark(tess_pred, gt)}",
        f"Easy:   {mark(easy_pred, gt)}",
        f"CRNN:   {mark(crnn_pred, gt)}",
    ]
    ax.set_title("\n".join(lines), fontsize=8, family="monospace", loc="left")
    ax.axis("off")

plt.suptitle("Сравнение всех подходов на одинаковых изображениях", fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

---
# Часть 10. Итоговая сводная таблица

In [ ]:
# === ФИНАЛЬНАЯ СВОДНАЯ ТАБЛИЦА ===

all_results = []

# Лучший Tesseract
best_tess = tesseract_results[best_tess_key]
all_results.append({
    "Подход": f"Tesseract ({best_tess_key})",
    "Тип": "OCR",
    "Exact Match, %": f"{best_tess['exact_match']:.1f}",
    "Char Accuracy, %": f"{best_tess['char_accuracy']:.1f}",
    "mAP@50, %": "—",
    "Inference, мс": f"{best_tess['inference_ms']:.0f}",
    "Обучение": "Нет",
})

# Лучший EasyOCR
best_easy = easyocr_results[best_easy_key]
all_results.append({
    "Подход": f"EasyOCR ({best_easy_key})",
    "Тип": "OCR",
    "Exact Match, %": f"{best_easy['exact_match']:.1f}",
    "Char Accuracy, %": f"{best_easy['char_accuracy']:.1f}",
    "mAP@50, %": "—",
    "Inference, мс": f"{best_easy['inference_ms']:.0f}",
    "Обучение": "Нет",
})

# PaddleOCR
all_results.append({
    "Подход": "PaddleOCR",
    "Тип": "OCR",
    "Exact Match, %": f"{paddle_metrics['exact_match']:.1f}",
    "Char Accuracy, %": f"{paddle_metrics['char_accuracy']:.1f}",
    "mAP@50, %": "—",
    "Inference, мс": f"{paddle_elapsed:.0f}",
    "Обучение": "Нет",
})

# CRNN
all_results.append({
    "Подход": "CRNN (ResNet18 + BiLSTM)",
    "Тип": "Custom NN",
    "Exact Match, %": f"{crnn_metrics['exact_match']:.1f}",
    "Char Accuracy, %": f"{crnn_metrics['char_accuracy']:.1f}",
    "mAP@50, %": "—",
    "Inference, мс": f"{crnn_avg_time:.0f}",
    "Обучение": f"{CRNN_EPOCHS} эпох",
})

# YOLO experiments
for exp_name, info in yolo_results.items():
    short = exp_name.replace("YOLOv8n_", "")
    all_results.append({
        "Подход": f"YOLOv8n ({short})",
        "Тип": "Detection",
        "Exact Match, %": f"{info.get('exact_match', 0):.1f}",
        "Char Accuracy, %": f"{info.get('char_accuracy', 0):.1f}",
        "mAP@50, %": f"{info.get('map50', 0)*100:.1f}",
        "Inference, мс": f"{info.get('inference_ms', 0):.0f}",
        "Обучение": f"{info.get('train_time_min', 0):.0f} мин",
    })


# Красивый вывод в виде HTML-таблицы
html = "<h3>Итоговая сводная таблица: сравнение подходов</h3>"
html += '<table border="1" cellpadding="8" cellspacing="0" style="border-collapse: collapse; font-size: 13px;">'
html += "<tr style='background: #2c3e50; color: white;'>"
for col in all_results[0].keys():
    html += f"<th>{col}</th>"
html += "</tr>"

for i, row in enumerate(all_results):
    bg = "#ecf0f1" if i % 2 == 0 else "#ffffff"
    # Подсветка лучшего результата
    is_best = "transfer" in row["Подход"].lower() or "freeze" in row["Подход"].lower()
    if is_best:
        bg = "#d5f5e3"
    html += f"<tr style='background: {bg};'>"
    for val in row.values():
        html += f"<td>{val}</td>"
    html += "</tr>"

html += "</table>"
display(HTML(html))

In [ ]:
# === ИТОГОВАЯ ГИСТОГРАММА ===

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

names = [r["Подход"] for r in all_results]
short_names = [n.replace("YOLOv8n ", "").replace("Tesseract ", "Tess ") for n in names]
em_vals = [float(r["Exact Match, %"]) for r in all_results]
char_vals = [float(r["Char Accuracy, %"]) for r in all_results]

# Цвета по типу
color_map = {"OCR": "#e74c3c", "Custom NN": "#f39c12", "Detection": "#27ae60"}
colors = [color_map[r["Тип"]] for r in all_results]

x = np.arange(len(names))

# Exact Match
bars1 = ax1.barh(x, em_vals, color=colors, edgecolor="black", height=0.6)
ax1.set_yticks(x)
ax1.set_yticklabels(short_names, fontsize=9)
ax1.set_xlabel("Exact Match, %", fontsize=12)
ax1.set_title("Точность (полное совпадение показания)", fontsize=13)
ax1.set_xlim(0, 105)
ax1.invert_yaxis()
for bar, val in zip(bars1, em_vals):
    ax1.text(val + 1, bar.get_y() + bar.get_height()/2,
             f"{val:.1f}%", va="center", fontsize=10, fontweight="bold")

# Char Accuracy
bars2 = ax2.barh(x, char_vals, color=colors, edgecolor="black", height=0.6)
ax2.set_yticks(x)
ax2.set_yticklabels(short_names, fontsize=9)
ax2.set_xlabel("Char Accuracy, %", fontsize=12)
ax2.set_title("Посимвольная точность", fontsize=13)
ax2.set_xlim(0, 105)
ax2.invert_yaxis()
for bar, val in zip(bars2, char_vals):
    ax2.text(val + 1, bar.get_y() + bar.get_height()/2,
             f"{val:.1f}%", va="center", fontsize=10, fontweight="bold")

# Легенда
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, edgecolor="black", label=l)
                   for l, c in color_map.items()]
ax1.legend(handles=legend_elements, loc="lower right", fontsize=10)

plt.suptitle("Итоговое сравнение подходов к распознаванию показаний счётчиков",
             fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# === ВРЕМЯ INFERENCE ===

fig, ax = plt.subplots(figsize=(12, 6))

inf_vals = [float(r["Inference, мс"]) for r in all_results]

bars = ax.barh(x, inf_vals, color=colors, edgecolor="black", height=0.6)
ax.set_yticks(x)
ax.set_yticklabels(short_names, fontsize=10)
ax.set_xlabel("Inference, мс (T4 GPU)", fontsize=12)
ax.set_title("Время обработки одного изображения", fontsize=14)
ax.invert_yaxis()
ax.axvline(x=300, color="red", linestyle="--", linewidth=2, label="Лимит для мобильного (300 мс)")

for bar, val in zip(bars, inf_vals):
    ax.text(val + 5, bar.get_y() + bar.get_height()/2,
            f"{val:.0f} мс", va="center", fontsize=10)

ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

---
# Часть 11. Выводы

## Результаты экспериментов

### OCR-движки без обучения:
- ❌ **Tesseract** — захватывает лишний текст (серийные номера, kWh), путает похожие цифры (1↔7, 6↔8)
- ❌ **EasyOCR** — лучше Tesseract, но всё равно не может отличить показание от остального текста на счётчике
- ❌ **PaddleOCR** — аналогичные проблемы: нет понимания контекста «что является показанием»

### Обучаемые модели:
- ⚠️ **CRNN + CTC** — обучается на показаниях, но требует точного кропа области дисплея. На произвольных фото — низкая точность.
- ✅ **YOLOv8n (scratch)** — работает, но долго обучается и хуже генерализуется
- ✅✅ **YOLOv8n (full fine-tune)** — хорошие результаты, но дольше и выше риск переобучения
- ✅✅✅ **YOLOv8n (transfer, freeze=10)** — лучший баланс: быстрое обучение, высокая точность, стабильность

### Итог: выбран **YOLOv8n с Transfer Learning (freeze=10)** как оптимальный подход.

In [ ]:
# Сохранение лучшей модели
best_name = max(yolo_results, key=lambda k: yolo_results[k].get("exact_match", 0))
best_path = yolo_results[best_name]["best_path"]
print(f"Лучшая модель: {best_name}")
print(f"Путь: {best_path}")
print(f"mAP@50: {yolo_results[best_name].get('map50', 0)*100:.1f}%")
print(f"Exact Match: {yolo_results[best_name].get('exact_match', 0):.1f}%")
print()
print("Модель готова для экспорта в TFLite через ноутбук meter_ocr_training.ipynb")